# **USO COBERTURA DE LA TIERRA**

In [ ]:
# librerias
import geopandas as gpd
import matplotlib.pyplot as plt
import os
import matplotlib.patches as mpatches

In [ ]:
# Cargar los shapefiles que va trabajar
cuenca_acacias = gpd.read_file("/content/drive/MyDrive/PyAgroCol/shp/cuenca_acacias/cuenca_acacias.shp")
uso_suelo = gpd.read_file("/content/drive/MyDrive/PyAgroCol/shp/shape coberturas 2018/cobertura_tierra_clc_2018.shp")


In [ ]:
uso_suelo = gpd.read_file("/content/drive/MyDrive/PyAgroCol/shp/shape coberturas 2018/cobertura_tierra_clc_2018.shp")

In [ ]:
# Asegurarse de que ambos shapefiles estén en el mismo sistema de coordenadas
if cuenca_acacias.crs != uso_suelo.crs:
    uso_suelo = uso_suelo.to_crs(cuenca_acacias.crs)

# Filtrar las etiquetas de los cultivos según la columna 'leyenda'
etiquetas_filtradas = ["2.1.5.1. Papa", '2.4.1. Mosaico de cultivos',
'2.4.3. Mosaico de cultivos, pastos y espacios naturales',
'2.4.4. Mosaico de pastos con espacios naturales',
'2.4.5. Mosaico de cultivos con espacios naturales',
'2.1.1. Otros cultivos transitorios', '2.1.2. Cereales', '2.1.2.1. Arroz',
'2.2.1.1. Otros cultivos permanentes herbáceos',
'2.2.2. Cultivos permanentes arbustivos', '2.2.3.2. Palma de aceite',
'2.1.2.3. Sorgo', '2.2.1. Cultivos permanentes herbáceos', '2.1.2.2. Maíz',
'2.1.3.4. Soya', '2.2.1.2. Caña','2.2.3.1. Otros cultivos permanentes arbóreos', '2.1.4.1. Cebolla']
uso_suelo_cultivos = uso_suelo[uso_suelo['leyenda'].isin(etiquetas_filtradas)]

#Intersectar la información de los cultivos
cultivos_orinoquia = gpd.overlay(uso_suelo_cultivos, cuenca_acacias, how='intersection')

# Removiendo la numeración al principio de cada etiqueta
cultivos_orinoquia['leyenda'] = cultivos_orinoquia['leyenda'].str.replace(r'^\d+(\.\d+)*\. ', '', regex=True)

# Visualizar
fig, ax = plt.subplots(figsize=(10, 10))  # Ajusta el tamaño de la figura
cuenca_acacias.boundary.plot(ax=ax, linewidth=1, color='black', label='Cuencas')
cultivos_orinoquia.boundary.plot(ax=ax, linewidth=0.5, color='green', label='Uso de suelo')
ax.axis('off')  # Desactivar ejes
ax.grid(False)  # Quitar las grillas
ax.legend(loc='upper right', fontsize=12)  # Personalizar la leyenda
plt.show() # Mostrar la figura

In [ ]:
# Guardar mapa
fig.savefig("/content/drive/MyDrive/PyAgroCol/mapas/uso_suelo_cuenca_acacias.png", dpi=300, bbox_inches='tight', pad_inches=0.5, transparent=True)

In [ ]:
# Calcular el área en km2
# CRS específico para Colombia, zona Orinoquía (MAGNA-SIRGAS / Colombia Bogota)
cultivos_orinoquia = cultivos_orinoquia.to_crs(epsg=3116)
cultivos_orinoquia['m2'] = cultivos_orinoquia['geometry'].area
cultivos_orinoquia['km2'] = cultivos_orinoquia['m2'] / 1e6
cultivos_orinoquia['ha'] = cultivos_orinoquia['m2'] / 1e4

# Agrupar por 'leyenda' y 'NOM_ZH' y sumar las áreas en km2
df_agrupado = cultivos_orinoquia.groupby(['leyenda', 'Subcuenca'])['ha'].sum().reset_index()

#Guardar los datos en un excel
if not os.path.exists('/content/drive/MyDrive/PyAgroCol/datos'): os.makedirs('/content/drive/MyDrive/PyAgroCol/datos')
df_agrupado.to_excel("/content/drive/MyDrive/PyAgroCol/datos/cultivos_cuenca_acacias.xlsx", index=False)

# Asegurarse de que ambos shapefiles estén en el mismo sistema de coordenadas
if cuenca_acacias.crs != cultivos_orinoquia.crs:
    cultivos_orinoquia = cultivos_orinoquia.to_crs(cuenca_acacias.crs)

In [ ]:
# Visualizar para reconocer cultivos
fig, ax = plt.subplots(figsize=(20, 20))  # Aumentar el tamaño del gráfico
cuenca_map = cultivos_orinoquia.plot(ax=ax, column='leyenda', cmap='tab20', legend=True,
    legend_kwds={
        'bbox_to_anchor': (1, 0.5),  # Ajustar la posición de la leyenda
        'loc': 'center left',
        'ncol': 1,
        'fontsize': 8,  # Tamaño de la fuente de la leyenda
        'title': "Leyenda"})
cuenca_acacias.boundary.plot(ax=ax, linewidth=0.5, color='black', label='Cuencas')
ax.axis('off')  # Desactivar ejes
leg = ax.get_legend()
title = leg.get_title()
title.set_weight('bold')
title.set_fontsize(10)  # Tamaño de la fuente del título de la leyenda
fig.tight_layout(rect=[0, 0, 0.85, 1])  # Ajustar el layout
plt.show()

fig.savefig("/content/drive/MyDrive/PyAgroCol/mapas/cultivos_cuenca_acacias.png", dpi=300, bbox_inches='tight', pad_inches=0.5, transparent=True)

if not os.path.exists('/content/drive/MyDrive/PyAgroCol/shp/cultivos_cuenca_acacias'): os.makedirs("/content/drive/MyDrive/PyAgroCol/shp/cultivos_cuenca_acacias")
cultivos_orinoquia.to_file("/content/drive/MyDrive/PyAgroCol/shp/cultivos_cuenca_acacias/cultivos_cuenca_acacias.shp")

# **FRONTERA AGRÍCOLA**

In [ ]:
###############################################################################
###  CORTAR LOS DATOS DE FRONTERA AGRICOLA EN LA CUENCA DEL RIO ACACIAS
###############################################################################

# Cargar los shapefiles que va trabajar
cuenca_acacias = gpd.read_file("shp/cuenca_acacias/cuenca_acacias.shp")
frontera_agricola_nacional = gpd.read_file("shp/Frontera_Agricola_May_2023/Frontera_Agricola_May_2023.shp")

# Asegurarse de que ambos shapefiles estén en el mismo sistema de coordenadas
if cuenca_acacias.crs != frontera_agricola_nacional.crs:
    cuenca_acacias = cuenca_acacias.to_crs(frontera_agricola_nacional.crs)

#Intersectar la información de los cultivos
frontera_cuenca_acacias = gpd.overlay(frontera_agricola_nacional, cuenca_acacias, how='intersection')



In [ ]:
#Eliminar columnas innecesarias
#frontera_cuenca_acacias.drop(['municipio', 'departamen', 'cod_dane_m', 'area_ha',
       'cod_depart', 'consecutiv', 'shape_Leng', 'shape_Area'], axis=1, inplace=True)

# Cambiar el nombre del nivel "Antiguo" a "Nuevo"
frontera_cuenca_acacias['elemento'] = frontera_cuenca_acacias['elemento'].replace('Frontera agricola nacional', 'Frontera agrícola nacional')

# Calcular el área en
# CRS específico para Colombia, zona Orinoquía (MAGNA-SIRGAS / Colombia Bogota)
frontera_cuenca_acacias = frontera_cuenca_acacias.to_crs(epsg=3116)
frontera_cuenca_acacias['m2'] = frontera_cuenca_acacias['geometry'].area
frontera_cuenca_acacias['km2'] = frontera_cuenca_acacias['m2'] / 1e6
frontera_cuenca_acacias['ha'] = frontera_cuenca_acacias['m2'] / 1e4

# Agrupar por 'leyenda' y 'NOM_ZH' y sumar las áreas en km2
df_agrupado = frontera_cuenca_acacias.groupby(['elemento', 'Subcuenca'])['ha'].sum().reset_index()

df_agrupado.to_excel("datos/frontera_cuenca_acacias.xlsx", index =False)



In [ ]:
# Visualizar para reconocer cultivos
fig, ax = plt.subplots(figsize=(14, 14))
color_map = {'Bosques naturales y áreas no agropecuarias': '#00FF00',
    'Frontera agrícola nacional': '#006400',
    'Exclusiones legales': '#808080'}
for elemento, color in color_map.items():
    frontera_cuenca_acacias[frontera_cuenca_acacias['elemento'] == elemento].plot(ax=ax, color=color, label=elemento)
cuenca_acacias.boundary.plot(ax=ax, linewidth=0.5, color='black')
legend_patches = [mpatches.Patch(color=color, label=elemento) for elemento, color in color_map.items()]
ax.legend(handles=legend_patches, bbox_to_anchor=(0.95, 0.99), loc='best', title="Leyenda")
leg = ax.get_legend()
title = leg.get_title()
title.set_weight('bold')
title.set_fontsize(10)
ax.axis('off')
fig.tight_layout(rect=[0, 0, 0.85, 0.5])
plt.show()

#Guargar
fig.savefig("mapas/frontera_cuenca_acacias.png", dpi=300, bbox_inches='tight', pad_inches=0.5, transparent=True)

if not os.path.exists('shp/frontera_cuenca_acacias'): os.makedirs("shp/frontera_cuenca_acacias")
frontera_cuenca_acacias.to_file("shp/frontera_cuenca_acacias/frontera_cuenca_acacias.shp")